# Merging SI poplave data

## Setup


In [1]:
!which python
# !pip install nbformat pandas plotly requests
# !pip install nbformat --upgrade
# !pip install plotly

/Users/klemenkubelj/miniconda3/envs/cvar-masters/bin/python


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import geopandas as gpd
import pandas as pd

## Define paths

In [16]:
RAW_DATA_DIR = "/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December"
HAND_CLEANED_DATA_DIR = "/Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin"

In [6]:
# List all directories in the data path
all_raw_dirs = [p for p in os.listdir(RAW_DATA_DIR) if os.path.isdir(os.path.join(RAW_DATA_DIR, p)) and p.startswith("OPVP")]
for d in sorted(all_raw_dirs):
    print(d)
print("len(all_raw_dirs): ", len(all_raw_dirs))

OPVP Begunje na Gorenjskem
OPVP Cerkno
OPVP Črna na Koroškem
OPVP Hrastovec
OPVP Izola
OPVP Kostanjevica
OPVP Kresnice
OPVP Kropa
OPVP Krška vas
OPVP Litija
OPVP Miren
OPVP Odranci
OPVP Ortnek
OPVP Piran
OPVP Radeče
OPVP Solčava
OPVP Tržič
OPVP Zagorje ob Savi
len(all_raw_dirs):  18


-----------

# Add prj files

In [7]:
epsg_3912 = """
PROJCS["MGI_1901_Slovene_National_Grid",GEOGCS["GCS_MGI_1901",DATUM["D_MGI_1901",SPHEROID["Bessel_1841",6377397.155,299.1528128]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["False_Easting",500000.0],PARAMETER["False_Northing",-5000000.0],PARAMETER["Central_Meridian",15.0],PARAMETER["Scale_Factor",0.9999],PARAMETER["Latitude_Of_Origin",0.0],UNIT["Meter",1.0]]
""".replace("\n", "")

def add_prj_file_to_shapefile(shapefile_path, force_recreate=True):
    if not os.path.exists(shapefile_path):
        raise FileNotFoundError(f"Shapefile not found: '{shapefile_path}'")
    if not shapefile_path.endswith(".shp"):
        raise ValueError(f"Shapefile path must end with '.shp': '{shapefile_path}'")
    
    # Check if prj file exists
    prj_file = shapefile_path.replace(".shp", ".prj")
    if os.path.exists(prj_file):
        if force_recreate:
            os.remove(prj_file)
        else:
            # print(f"Prj file already exists for '{shapefile_path}'")
            return False
    
    # Create prj file
    with open(prj_file, "w") as f:
        f.write(epsg_3912)
    return True


In [14]:
# Iterate over all files in the directory
stats = {
    "new": 0,
    "existing": 0,
}
for d in all_raw_dirs:
    dir_path = os.path.join(RAW_DATA_DIR, d)
    # walk through the directory
    for root, dirs, files in os.walk(dir_path):
        for f in files:
            if f.endswith(".shp"):
                shp_file = os.path.join(root, f)
                added = add_prj_file_to_shapefile(shp_file, force_recreate=False)
                if added:
                    stats["new"] += 1
                else:
                    stats["existing"] += 1

print("Added", stats["new"], "prj files")
print("Existing", stats["existing"], "prj files")


Added 0 prj files
Existing 226 prj files


# Structured shapefile tree

In [36]:
def is_arcgis_format(shapefile_path):
    # Check for ArcGIS-specific files
    # arcgis_extensions = [".sbn", ".sbx", ".fbn", ".fbx", ".ain", ".aih", ".ixs", ".mxs", ".atx", ".cpg", ".xml", ".mxd"]
    # for ext in arcgis_extensions:
    #     if os.path.exists(shapefile_path.replace(".shp", ext)):
    #         return True
    
    # Check for "arcgis" in the file path
    if "arcgis" in shapefile_path.lower():
        return True
    
    return False


In [37]:
p = '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Izola/3510.shp'
is_arcgis_format(p)

False

In [45]:
# QGIS format
qgis_stats = {
    # region_name {
    #     "q10": 0,
    #     "q100": 0,
    #     "q500": 0,
    # }
}
qgis_format_regions = []
# iterate over all shapefiles in /Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin/q10, /Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin/q100, /Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin/q500
for rp in ["q10", "q100", "q500"]:
    for f in os.listdir(os.path.join(HAND_CLEANED_DATA_DIR, rp)):
        if f.endswith(".shp"):
            region = f.split("_")[0]
            if region not in qgis_stats:
                qgis_stats[region] = {
                    "q10": 0,
                    "q100": 0,
                    "q500": 0,
                }
            qgis_stats[region][rp] += 1
        qgis_format_regions.append(region)

qgis_format_regions = sorted(list(set(qgis_format_regions)))
assert len(qgis_format_regions) == len(qgis_stats)

print("There are ", len(qgis_format_regions), " QGIS format regions")
print("There are ", sum([i["q10"] for i in qgis_stats.values()]), " shapefiles for Q10")
print("There are ", sum([i["q100"] for i in qgis_stats.values()]), " shapefiles for Q100")
print("There are ", sum([i["q500"] for i in qgis_stats.values()]), " shapefiles for Q500")
qgis_stats



There are  10  QGIS format regions
There are  30  shapefiles for Q10
There are  30  shapefiles for Q100
There are  30  shapefiles for Q500


{'miren': {'q10': 3, 'q100': 3, 'q500': 3},
 'zagorje': {'q10': 3, 'q100': 3, 'q500': 3},
 'radece': {'q10': 3, 'q100': 3, 'q500': 3},
 'kostanjevica': {'q10': 3, 'q100': 3, 'q500': 3},
 'odranci': {'q10': 3, 'q100': 3, 'q500': 3},
 'orntek': {'q10': 3, 'q100': 3, 'q500': 3},
 'solcava': {'q10': 3, 'q100': 3, 'q500': 3},
 'kresnice': {'q10': 3, 'q100': 3, 'q500': 3},
 'krska': {'q10': 3, 'q100': 3, 'q500': 3},
 'litija': {'q10': 3, 'q100': 3, 'q500': 3}}

In [49]:
# ArcGis format
arcgis_format_regions = []
for f in os.listdir(os.path.join(HAND_CLEANED_DATA_DIR, "arcgis")):
    if f.endswith(".shp"):
        region = f.split("_")[0]
        arcgis_format_regions.append(region)

arcgis_format_regions = sorted(arcgis_format_regions)
print("There are ", len(arcgis_format_regions), " ArcGis format regions")

There are  8  ArcGis format regions


In [52]:
# Sanity check, make sure all regions are accounted for
assert len(arcgis_format_regions) + len(qgis_format_regions) == len(all_raw_dirs)
print("All regions are accounted for, there are ", len(all_raw_dirs), " regions")

All regions are accounted for, there are  18  regions


----------

# Processing

## Part 1: QGIS shapefiles
Shapefiles split by return period and intensity

In [17]:
HAND_CLEANED_DATA_DIR = "/Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin"
q10_dir = os.path.join(HAND_CLEANED_DATA_DIR, "q10")
q100_dir = os.path.join(HAND_CLEANED_DATA_DIR, "q100")
q500_dir = os.path.join(HAND_CLEANED_DATA_DIR, "q500")
q10_shapefiles = [os.path.join(q10_dir, f) for f in os.listdir(q10_dir) if f.endswith(".shp")]
q100_shapefiles = [os.path.join(q100_dir, f) for f in os.listdir(q100_dir) if f.endswith(".shp")]
q500_shapefiles = [os.path.join(q500_dir, f) for f in os.listdir(q500_dir) if f.endswith(".shp")]
all_shapefiles = q10_shapefiles + q100_shapefiles + q500_shapefiles
print("Found ", len(all_shapefiles), " shapefiles")
print("q10: ", len(q10_shapefiles))
print("q100: ", len(q100_shapefiles))
print("q500: ", len(q500_shapefiles))


Found  89  shapefiles
q10:  30
q100:  30
q500:  29


In [88]:
def _get_globina(row):
    source = row["k_source_file"].lower()
    if source.endswith("gv.shp"):
        return 2.5
    elif source.endswith("gs.shp"):
        return 1.5
    elif source.endswith("gm.shp"):
        return 0.5
    raise ValueError(f"Unknown source: '{source}'")

def _clean_shapefile(_df, path):
    # Relative to DATA_DIR
    RELDIR = "/Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin"
    source_file = os.path.relpath(path, RELDIR)
    _df["k_source_file"] = source_file
    _df["k_region"] = source_file.split("/")[-1].split("_Q")[0] # Extract source from path
    _df["k_globina"] = _df.apply(lambda row: _get_globina(row), axis=1)
    return _df

def _read_shapefile(path):
    _df = gpd.read_file(path)
    # Check if _df is empty
    if _df.empty:
        print("Empty shapefile: ", path, ". Skipping...")
        return None
    _df = _clean_shapefile(_df, path)
    return _df

dfs = []
for path in all_shapefiles:
    _df = _read_shapefile(path)
    if _df is not None:
        dfs.append(_df)

print("Read ", len(dfs), " shapefiles")

Empty shapefile:  /Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin/q10/orntek_Q10gm.shp . Skipping...
Empty shapefile:  /Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin/q10/odranci_Q10gv.shp . Skipping...
Empty shapefile:  /Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin/q10/orntek_Q10gv.shp . Skipping...
Read  86  shapefiles


In [89]:
# Combine all dataframes into one
df_all = pd.concat(dfs)

In [92]:
df_all.head()

,Shape_Leng,Shape_Area,geometry,k_source_file,k_region,k_globina,ID,OZN_K,UREDB_PRAV,DIM_HID_M,...,IME_IZD,OPOMBA,Id,area_m2,GRIDCODE,A,BUFF_DIST,Shape_Le_1,OBJECTID,Shape_Le_2
0,160.982643,274.361222,"POLYGON ((391200.136 83472.451, 391212.731 834...",q10/miren_Q10gs.shp,miren,1.5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1674.851268,68455.737056,"POLYGON ((391231.886 83871.331, 391237.532 838...",q10/miren_Q10gs.shp,miren,1.5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,217.969449,383.311844,"POLYGON ((391367.093 83495.968, 391319.734 834...",q10/miren_Q10gs.shp,miren,1.5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,106.922810,260.697746,"POLYGON ((391064.228 83574.222, 391057.072 835...",q10/miren_Q10gs.shp,miren,1.5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,157.975100,1336.159044,"POLYGON ((391596.480 83387.798, 391595.080 833...",q10/miren_Q10gs.shp,miren,1.5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [94]:
# Uniqu values of k_source_file
df_all["k_globina"].unique()

array([1.5, 2.5, 0.5])

In [61]:
# # To shapefile
# rad_q10 = '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Radeče/shape/Q10g_MERGED.shp'
# df_rad_q10.to_file(rad_q10, driver="ESRI Shapefile")


# Import ArcGis shapefiles

In [67]:
ARCGIS_DIR = "/Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin/arcgis"
arcgis_shapefiles = [os.path.join(ARCGIS_DIR, f) for f in os.listdir(ARCGIS_DIR) if f.endswith(".shp")]
print("Found ", len(arcgis_shapefiles), " ArcGis shapefiles")

Found  8  ArcGis shapefiles


In [119]:
def _get_globina_and_rp_arcgis(row) -> tuple[str, float]:
    layer = row["Layer"].lower()
    _map = {
        "q100_globina nad 15m": ("q100", 2.5),
        "q500_globina med 05_15m": ("q500", 1.5),
    }
    skip_layers = [
        "q100_v x h med 05_15m2s",
        "q500_doseg",
    ]
    if layer in _map:
        return _map[layer]
    if layer in skip_layers:
        return (None, None)
    return (None, None)
    raise ValueError(f"Unknown layer: '{layer}'")
    
def _clean_arcgis_shapefile(_df, path):
    # Relative to DATA_DIR
    RELDIR = "/Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin"
    source_file = os.path.relpath(path, RELDIR)
    _df["k_source_file"] = source_file
    _df["k_region"] = source_file.split("/")[-1].rsplit("_", 1)[0]
    # Set both "k_globina" and "k_rp" from same function "_get_globina_and_rp_arcgis"
    _df["k_globina"], _df["k_rp"] = zip(*_df.apply(lambda row: _get_globina_and_rp_arcgis(row), axis=1))
    return _df

def _read_arcgis_shapefile(path):
    _df = gpd.read_file(path)
    # Check if _df is empty
    if _df.empty:
        print("Empty shapefile: ", path, ". Skipping...")
        return None
    _df = _clean_arcgis_shapefile(_df, path)
    return _df

In [120]:
# Read all ArcGis shapefiles
arcgis_dfs = []
for path in arcgis_shapefiles:
    _df = _read_arcgis_shapefile(path)
    if _df is not None:
        arcgis_dfs.append(_df)
print("Read ", len(arcgis_dfs), " ArcGis shapefiles")

Read  8  ArcGis shapefiles


In [121]:
# Join all ArcGis dataframes into one
df_arcgis = pd.concat(arcgis_dfs)
df_arcgis.tail()

/var/folders/ly/jtbv8d056fx1pnpygvdlzn0r0000gn/T/ipykernel_3389/2317962754.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_arcgis = pd.concat(arcgis_dfs)


,FID_,Entity,Layer,Color,Linetype,Elevation,LineWt,RefName,st_vira,RPN,geometry,k_source_file,k_region,k_globina,k_rp,St_vira,enotna_sif
479,0,2DPolyline,razred globine_Q500gv,150,Continuous,0.0,20,None,3510,Q500gv,"POLYGON Z ((395736.447 45055.662 0.000, 395736...",arcgis/izola_3510.shp,izola,None,NaN,NaN,None
480,0,2DPolyline,OVR,10,Continuous,0.0,25,None,3510,OVR,"POLYGON Z ((396199.500 44472.495 0.000, 396067...",arcgis/izola_3510.shp,izola,None,NaN,NaN,None
481,0,LWPolyline,OHM,2,Continuous,0.0,25,None,3510,OHM,"POLYGON Z ((394800.000 43850.000 0.000, 394800...",arcgis/izola_3510.shp,izola,None,NaN,NaN,None
482,0,LWPolyline,RPN_vPv_(1D),1,BYBLOCK,0.0,20,None,3510,Pv,"POLYGON Z ((394833.500 44184.000 0.000, 394834...",arcgis/izola_3510.shp,izola,None,NaN,NaN,None
483,0,LWPolyline,razred globine_Q100gv,150,BYBLOCK,0.0,20,None,3510,Q100gv,"POLYGON Z ((394833.500 44184.000 0.000, 394834...",arcgis/izola_3510.shp,izola,None,NaN,NaN,None


In [122]:
# Unique values of "Layer" column
df_arcgis["Layer"].unique()

array(['Q100_v x h med 05_15m2s', 'Q100_globina nad 15m', 'Q500_doseg',
       'Q500_globina med 05_15m', 'Q10_doseg', 'Q10_globina med 05_15m',
       'Q100_globina med 05_15m', 'Q10_globina nad 15m',
       'Q10_globina pod 05m', 'Q100_doseg', 'Q100_globina pod 05m',
       'Q500_globina nad 15m', 'Q500_globina pod 05m',
       'Q100_v x h nad 15m2s', 'Q100_v x h pod 05m2s', 'Q100_razred_Pp',
       'Q100_razred_Pv', 'Q100_razred_Ps', 'Q100_razred_Pm', 'Struga',
       'Q100_hitrost veè 1ms', 'OHM', 'OVR', 'razred_Pv', 'razred_Ps',
       'razred_Pm', 'struga', 'razred_Pp', 'G10', 'G100', 'G500',
       'G10 gv', 'G10 gm', 'G10 gs', 'G100 gm', 'G100 gv', 'G100 gs',
       'G500 gv', 'G500 gm', 'G500 gs', 'Pv', 'Ps', 'Pm', 'Pp',
       'Obmoèje modela', 'Obmoèje veljavnosti rezultatov',
       '0_dosegi Q100', '0_dosegi Q10', '0_dosegi Q500', '0_Q10_h0.5',
       '0_Q100_h0.5', '0_Pm', '0_Q500_h0.5', '0_Q500_h0.5-1.5', '0_Ps',
       '0_Q500_h1.5', '0_Q10_h0.5-1.5', '0_Q100_h0.5-1.5',

In [12]:
### Try to import ArcGIS shapefiles
p = '/Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin_cleaned_raw/OPVP Tržič/ArcGis oblika/3466.shp'
df1 = gpd.read_file(p)

### Try to import ArcGIS shapefiles
p = '/Users/klemenkubelj/Documents/school/graduate/masters/data/si_karte_globin_cleaned_raw/OPVP Piran/3509.shp'
df2 = gpd.read_file(p)


In [62]:
df1.head()

,Shape_Leng,Shape_Area,geometry,k_source_file,k_region,k_globina
0,160.982643,274.361222,"POLYGON ((391200.136 83472.451, 391212.731 834...",q10/miren_Q10gs.shp,miren,1.5
1,1674.851268,68455.737056,"POLYGON ((391231.886 83871.331, 391237.532 838...",q10/miren_Q10gs.shp,miren,1.5
2,217.969449,383.311844,"POLYGON ((391367.093 83495.968, 391319.734 834...",q10/miren_Q10gs.shp,miren,1.5
3,106.922810,260.697746,"POLYGON ((391064.228 83574.222, 391057.072 835...",q10/miren_Q10gs.shp,miren,1.5
4,157.975100,1336.159044,"POLYGON ((391596.480 83387.798, 391595.080 833...",q10/miren_Q10gs.shp,miren,1.5


In [14]:
# Unique values in "Layer" column
df2["Layer"].unique()


array(['G10', 'G100', 'G500', 'G10 gv', 'G10 gm', 'G10 gs', 'G100 gm',
       'G100 gv', 'G100 gs', 'G500 gv', 'G500 gm', 'G500 gs', 'Pv', 'Ps',
       'Pm', 'Pp', 'Obmoèje modela', 'Obmoèje veljavnosti rezultatov'],
      dtype=object)

## Part 1: QGIS shapefiles
Shapefiles split by return period and intensity


In [ ]:
shapefile_tree = {
    "q10": {
        "m": [],
        "s": [],
        "v": [],
    },
    "q100": {
        "m": [],
        "s": [],
        "v": [],
    },
    "q500": {
        "m": [],
        "s": [],
        "v": [],
    },
}
skipped = []
arcgis_skipped = []
def categorize_shapefiles(shapefile_path):
    rp = None # Return period
    intensity = None # Depth, either "m", "s", or "v"

    q100_keywords = ["q100", ]
    q500_keywords = ["q500", ]
    q10_keywords = ["q10", ]

    if any(keyword in shapefile_path.lower() for keyword in q100_keywords):
        rp = "q100"
    elif any(keyword in shapefile_path.lower() for keyword in q500_keywords):
        rp = "q500"
    elif any(keyword in shapefile_path.lower() for keyword in q10_keywords):
        rp = "q10"

    v_keywords = ["gv", "globina_vec_od_1.5"]
    s_keywords = ["gs", "globina_vec_od_1"]
    m_keywords = ["gm", "globina_vec_od_0.5"]

    if any(keyword in shapefile_path.lower() for keyword in v_keywords):
        intensity = "v"
    elif any(keyword in shapefile_path.lower() for keyword in s_keywords):
        intensity = "s"
    elif any(keyword in shapefile_path.lower() for keyword in m_keywords):
        intensity = "m"

    return rp, intensity

# Go into each directory (nested, if there are nested directories) and find all .shp files.
# Categorize them by return period and intensity and add them to the shapefile tree.
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        if f.endswith(".shp"):
            shapefile_path = os.path.join(root, f)
            if is_arcgis_format(shapefile_path):
                arcgis_skipped.append(shapefile_path)
                continue

            rp, intensity = categorize_shapefiles(shapefile_path)
            if rp and intensity:
                shapefile_tree[rp][intensity].append(shapefile_path)
            else:
                skipped.append(shapefile_path)


In [ ]:
shapefile_tree

{'q10': {'m': ['/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Zagorje ob Savi/SHP_KPN_Q10/Q10gm.shp',
   '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Radeče/shape/Q10gm.shp',
   '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Kostanjevica/KPN/Q10gm.shp',
   '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Solčava/shp_Q10/Q10gm.shp',
   '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Miren/globine/Q10gm.shp',
   '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Krška vas/shape/Q10gm.shp',
   '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Odranci/OPVP_Odranci_SHP_visine/Q10gm_VisinaPopvode_10let_0_5m_veljavnost.shp'],
  's': ['/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Glo

In [ ]:
skipped

['/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Izola/3510.shp',
 '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Zagorje ob Savi/SHP_KPN_Q10/Q10.shp',
 '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Zagorje ob Savi/SHP_KPN_Q10/Q10vv1.shp',
 '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Zagorje ob Savi/SHP_KPN_Q100/Q10.shp',
 '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Zagorje ob Savi/SHP_KPN_Q100/Q500.shp',
 '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Zagorje ob Savi/SHP_KPN_Q100/Q100.shp',
 '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Zagorje ob Savi/SHP_KPN_Q100/vv1.shp',
 '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Zagorje ob Savi/SH

In [ ]:
arcgis_skipped

['/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Tržič/ArcGis oblika/3466.shp',
 '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Hrastovec/ArcGis oblika/3467.shp',
 '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Črna na Koroškem/ArcGis oblika/3464.shp',
 '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Begunje na Gorenjskem/ArcGis oblika/3465.shp',
 '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Kropa/ArcGis oblika/3462.shp',
 '/Users/klemenkubelj/Documents/school/graduate/masters/data/Karte Globine - December/OPVP Cerkno/ArcGis oblika/3463.shp']